# Pricing Strategy Analysis

This notebook focuses on analyzing pricing strategies and their impact including:
1. Price Elasticity
2. Discount Impact Analysis
3. Category Pricing Analysis
4. Revenue Optimization

## Setup and Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

plt.style.use('seaborn')
%matplotlib inline

## 1. Data Loading and Price Metrics

In [ ]:
# Load the data
df = pd.read_csv('../data/raw/sales_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Calculate price metrics
price_metrics = df.groupby('Product').agg({
    'Price': ['mean', 'min', 'max', 'std'],
    'Quantity': 'sum',
    'Total_Amount': 'sum',
    'Discount': 'mean'
}).round(2)

price_metrics.columns = ['avg_price', 'min_price', 'max_price', 'price_std',
                        'total_quantity', 'total_revenue', 'avg_discount']

price_metrics.head()

## 2. Price Distribution Analysis

In [ ]:
# Price distribution by category
plt.figure(figsize=(12, 6))
sns.violinplot(x='Category', y='Price', data=df)
plt.title('Price Distribution by Category')
plt.xticks(rotation=45)
plt.show()

# Price points heatmap
price_category = pd.crosstab(pd.qcut(df['Price'], q=5), df['Category'])
plt.figure(figsize=(10, 6))
sns.heatmap(price_category, cmap='YlOrRd', annot=True, fmt='d')
plt.title('Price Points Distribution Across Categories')
plt.show()

## 3. Discount Impact Analysis

In [ ]:
# Calculate discount effectiveness
df['discount_percentage'] = (df['Discount'] / (df['Price'] * df['Quantity'])) * 100

# Analyze sales volume vs discount
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='discount_percentage', y='Quantity')
plt.title('Impact of Discounts on Sales Volume')
plt.xlabel('Discount Percentage')
plt.ylabel('Quantity Sold')
plt.show()

# Average quantity sold by discount bracket
df['discount_bracket'] = pd.qcut(df['discount_percentage'], q=5)
discount_impact = df.groupby('discount_bracket')['Quantity'].mean()

plt.figure(figsize=(10, 6))
discount_impact.plot(kind='bar')
plt.title('Average Quantity Sold by Discount Bracket')
plt.xlabel('Discount Bracket')
plt.ylabel('Average Quantity Sold')
plt.xticks(rotation=45)
plt.show()

## 4. Price Elasticity Analysis

In [ ]:
# Calculate price elasticity by category
def calculate_elasticity(group):
    # Calculate percentage changes
    price_pct_change = group['Price'].pct_change()
    quantity_pct_change = group['Quantity'].pct_change()
    
    # Calculate elasticity
    elasticity = (quantity_pct_change / price_pct_change).mean()
    return elasticity

elasticity_by_category = df.groupby('Category').apply(calculate_elasticity)

# Plot elasticity
plt.figure(figsize=(10, 6))
elasticity_by_category.plot(kind='bar')
plt.title('Price Elasticity by Category')
plt.xlabel('Category')
plt.ylabel('Price Elasticity')
plt.xticks(rotation=45)
plt.show()

## 5. Revenue Optimization Analysis

In [ ]:
# Revenue by price point
df['price_bracket'] = pd.qcut(df['Price'], q=5)
revenue_by_price = df.groupby('price_bracket')['Total_Amount'].sum()

plt.figure(figsize=(10, 6))
revenue_by_price.plot(kind='bar')
plt.title('Revenue by Price Bracket')
plt.xlabel('Price Bracket')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.show()

# Optimal price points
optimal_prices = df.groupby('Product').apply(lambda x: {
    'optimal_price': x.groupby('Price')['Total_Amount'].sum().idxmax(),
    'max_revenue': x.groupby('Price')['Total_Amount'].sum().max()
}).apply(pd.Series)

print("Optimal Price Points by Product:")
optimal_prices.sort_values('max_revenue', ascending=False).head(10)

## 6. Key Insights

1. Price Sensitivity:
   - Identified price elastic and inelastic categories
   - Analyzed optimal price points by product

2. Discount Effectiveness:
   - Measured impact of discounts on sales volume
   - Identified optimal discount levels

3. Category Pricing:
   - Analyzed price distributions by category
   - Identified pricing opportunities

4. Revenue Optimization:
   - Found revenue-maximizing price points
   - Analyzed price-revenue relationships

5. Recommendations:
   - Category-specific pricing strategies
   - Optimal discount levels
   - Revenue optimization opportunities